# Module 2: Reward, Value, and the Bellman Equation

**Spinning Up in Active Inference** | Notebook 2 of 16

---

In Module 1 we saw agents learn from reward using Q-learning — but we treated the algorithm as a black box. In this module we open the box. We will derive the **Bellman equation**, the recursive relationship that makes all of RL possible, and implement **value iteration** from scratch.

The Bellman equation answers: *"If I act optimally from now on, how much total reward can I expect from state s?"* Active Inference asks a parallel question: *"If I follow policy pi from now on, how much surprise (free energy) will I accumulate?"* Both are recursive, both evaluate futures — but they optimize different objectives.

By the end of this notebook you will:
1. Understand Markov Decision Processes (MDPs) as the formal framework for sequential decision-making
2. Derive and implement the Bellman equation
3. Implement value iteration and policy extraction from scratch
4. See how discount factor gamma shapes the value landscape

**Prerequisites:** Module 1, basic linear algebra.  
**Time:** ~60 minutes.

## 2.1 Markov Decision Processes

A **Markov Decision Process** (MDP) is defined by the tuple $(\mathcal{S}, \mathcal{A}, P, R, \gamma)$:

- $\mathcal{S}$: Set of **states** (e.g., grid cells)
- $\mathcal{A}$: Set of **actions** (e.g., up/down/left/right)
- $P(s' | s, a)$: **Transition function** — probability of landing in $s'$ after taking action $a$ in state $s$
- $R(s, a, s')$: **Reward function** — immediate payoff for transitioning from $s$ to $s'$ via action $a$
- $\gamma \in [0, 1)$: **Discount factor** — how much we value future rewards relative to immediate ones

The **Markov property** states that the future depends only on the current state, not on how we got there:

$$P(s_{t+1} | s_t, a_t, s_{t-1}, a_{t-1}, \ldots) = P(s_{t+1} | s_t, a_t)$$

This is a strong assumption — but it is exactly what makes dynamic programming possible.

## 2.2 The Value Function and the Bellman Equation

The **state-value function** $V^\pi(s)$ is the expected cumulative discounted reward starting from state $s$ and following policy $\pi$:

$$V^\pi(s) = \mathbb{E}_\pi \left[ \sum_{t=0}^{\infty} \gamma^t r_t \mid s_0 = s \right]$$

The **Bellman equation** decomposes this into an immediate reward plus the discounted value of the next state:

$$V^\pi(s) = \sum_a \pi(a|s) \sum_{s'} P(s'|s,a) \left[ R(s,a,s') + \gamma V^\pi(s') \right]$$

For the **optimal** value function $V^*(s)$, we take the maximum over actions instead of averaging:

$$\boxed{V^*(s) = \max_a \sum_{s'} P(s'|s,a) \left[ R(s,a,s') + \gamma V^*(s') \right]}$$

This is the **Bellman optimality equation**. It says: *the value of a state under the optimal policy equals the best action's immediate reward plus the discounted value of where that action takes you.*

This recursive structure is what makes **dynamic programming** work — we can compute $V^*$ by iterating this equation until convergence.

## 2.3 The Reward Hypothesis

RL rests on what Sutton calls the **reward hypothesis**:

> *"All of what we mean by goals and purposes can be well thought of as the maximization of the expected value of the cumulative sum of a received scalar signal (called reward)."*
> — Sutton & Barto (2018)

This is a remarkably strong claim. Can hunger, curiosity, social bonding, and aesthetic appreciation all be reduced to a single scalar? Active Inference offers an alternative: organisms do not maximize reward — they minimize **surprise** (or equivalently, free energy). Reward is just one observation that the organism has learned to prefer. We will develop this alternative view over the coming modules.

For now, let us take the reward hypothesis at face value and see how far it gets us.

In [ ]:
# ── Setup ──────────────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from neuronav.envs.grid_env import GridEnv, GridSize, GridObservation
from neuronav.envs.grid_templates import GridTemplate

import sys; sys.path.insert(0, '..')
from utils.plotting import (
    plot_value_heatmap, plot_learning_curve, plot_agent_trajectory,
    dual_perspective_box, COLORS
)

np.random.seed(42)
print("Imports successful!")

## 2.4 Building a Simple MDP by Hand

Before using neuro-nav, let us construct a tiny MDP from scratch — a 4x4 grid world — so we can see every component of the Bellman equation explicitly.

In [ ]:
# ── Define a 4x4 grid MDP manually ───────────────────────────────────────────

GRID_SIZE = 4
N_STATES = GRID_SIZE ** 2  # 16 states
N_ACTIONS = 4  # 0=up, 1=down, 2=left, 3=right
GOAL_STATE = 15  # bottom-right corner
GAMMA = 0.99

def state_to_pos(s):
    """Convert linear state index to (row, col)."""
    return (s // GRID_SIZE, s % GRID_SIZE)

def pos_to_state(r, c):
    """Convert (row, col) to linear state index."""
    return r * GRID_SIZE + c

# Build transition matrix P[s, a] -> s'
# For simplicity: deterministic transitions, stay in place if hitting a wall
P = np.zeros((N_STATES, N_ACTIONS, N_STATES))  # P[s, a, s'] = probability

action_deltas = {
    0: (-1, 0),  # up
    1: (1, 0),   # down
    2: (0, -1),  # left
    3: (0, 1),   # right
}

for s in range(N_STATES):
    r, c = state_to_pos(s)
    for a in range(N_ACTIONS):
        dr, dc = action_deltas[a]
        nr, nc = r + dr, c + dc
        # Boundary check
        if 0 <= nr < GRID_SIZE and 0 <= nc < GRID_SIZE:
            next_s = pos_to_state(nr, nc)
        else:
            next_s = s  # stay in place
        P[s, a, next_s] = 1.0

# Reward: +1 for reaching the goal, 0 everywhere else
R = np.zeros(N_STATES)
R[GOAL_STATE] = 1.0

print(f"MDP: {N_STATES} states, {N_ACTIONS} actions")
print(f"Goal state: {GOAL_STATE} = position {state_to_pos(GOAL_STATE)}")
print(f"P shape: {P.shape} (states x actions x states)")

In [ ]:
# ── Value Iteration from scratch ─────────────────────────────────────────────
# This is the core dynamic programming algorithm:
#   V(s) <- max_a sum_{s'} P(s'|s,a) * [R(s') + gamma * V(s')]
# Repeat until convergence.

def value_iteration(P, R, gamma=0.99, theta=1e-8, max_iters=1000):
    """Tabular value iteration.
    
    Args:
        P: Transition matrix, shape (n_states, n_actions, n_states)
        R: Reward vector, shape (n_states,) — reward for arriving at state
        gamma: Discount factor
        theta: Convergence threshold
        max_iters: Maximum iterations
    
    Returns:
        V: Optimal value function, shape (n_states,)
        history: List of V arrays at each iteration (for visualization)
    """
    n_states = P.shape[0]
    V = np.zeros(n_states)
    history = [V.copy()]
    
    for i in range(max_iters):
        V_new = np.zeros(n_states)
        for s in range(n_states):
            # Bellman optimality equation
            q_values = np.zeros(P.shape[1])
            for a in range(P.shape[1]):
                # Q(s,a) = sum_{s'} P(s'|s,a) * [R(s') + gamma * V(s')]
                q_values[a] = np.sum(P[s, a, :] * (R + gamma * V))
            V_new[s] = np.max(q_values)
        
        delta = np.max(np.abs(V_new - V))
        V = V_new
        history.append(V.copy())
        
        if delta < theta:
            print(f"Converged after {i+1} iterations (delta={delta:.2e})")
            break
    
    return V, history

V_star, vi_history = value_iteration(P, R, gamma=GAMMA)
print(f"\nOptimal V(s=0) = {V_star[0]:.4f}")
print(f"Optimal V(goal) = {V_star[GOAL_STATE]:.4f}")

In [ ]:
# ── Visualize value iteration convergence ────────────────────────────────────

fig, axes = plt.subplots(1, 4, figsize=(20, 4))
iterations_to_show = [0, 2, 10, len(vi_history)-1]

for ax, it in zip(axes, iterations_to_show):
    it = min(it, len(vi_history) - 1)
    plot_value_heatmap(
        vi_history[it], GRID_SIZE,
        title=f"Iteration {it}",
        cmap="viridis",
        goal_pos=state_to_pos(GOAL_STATE),
        ax=ax
    )

fig.suptitle("Value Iteration Convergence: Value Propagates Backwards from the Goal", 
             fontsize=14, y=1.05)
plt.tight_layout()
plt.show()

Notice how value "flows" backward from the goal. At iteration 0, only the goal state has value. By iteration 2, its neighbors have acquired some value. By convergence, every state has a value reflecting its distance from the goal, discounted by $\gamma$.

This backward propagation of value through state space is the computational essence of the Bellman equation.

In [ ]:
# ── Extract the optimal policy from V* ───────────────────────────────────────
# The optimal policy is: pi*(s) = argmax_a sum_{s'} P(s'|s,a) [R(s') + gamma V*(s')]

def extract_policy(V, P, R, gamma):
    """Extract greedy policy from value function."""
    n_states, n_actions = P.shape[0], P.shape[1]
    policy = np.zeros(n_states, dtype=int)
    for s in range(n_states):
        q_values = np.array([
            np.sum(P[s, a, :] * (R + gamma * V))
            for a in range(n_actions)
        ])
        policy[s] = np.argmax(q_values)
    return policy

optimal_policy = extract_policy(V_star, P, R, GAMMA)

# Visualize as arrows on the grid
action_arrows = {0: '\u2191', 1: '\u2193', 2: '\u2190', 3: '\u2192'}  # up, down, left, right

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Value function
plot_value_heatmap(
    V_star, GRID_SIZE,
    title="Optimal Value Function V*(s)",
    cmap="viridis",
    goal_pos=state_to_pos(GOAL_STATE),
    show_values=True,
    ax=ax1
)

# Policy as arrows
grid_display = V_star.reshape(GRID_SIZE, GRID_SIZE)
ax2.imshow(grid_display, cmap='viridis', interpolation='nearest')
for s in range(N_STATES):
    r, c = state_to_pos(s)
    if s == GOAL_STATE:
        ax2.text(c, r, '\u2605', ha='center', va='center', fontsize=20, color='gold')
    else:
        arrow = action_arrows[optimal_policy[s]]
        ax2.text(c, r, arrow, ha='center', va='center', fontsize=18, 
                color='white', fontweight='bold')
ax2.set_title("Optimal Policy \u03C0*(s)\n(arrows show best action)", fontsize=13)
ax2.set_xticks(range(GRID_SIZE))
ax2.set_yticks(range(GRID_SIZE))

plt.tight_layout()
plt.show()

## 2.5 The Discount Factor: How Far Ahead Do You Look?

The discount factor $\gamma$ controls the effective planning horizon:

- $\gamma \to 0$: Agent is **myopic** — only cares about immediate reward
- $\gamma \to 1$: Agent is **far-sighted** — values distant rewards almost as much as immediate ones

The effective horizon is approximately $\frac{1}{1 - \gamma}$ steps. So $\gamma = 0.5$ gives a horizon of ~2 steps, while $\gamma = 0.99$ gives ~100 steps.

In Active Inference, the planning horizon is explicit (parameter $T$), not encoded implicitly through discounting. This is one way the two frameworks differ in their treatment of time.

In [ ]:
# ── Compare value functions with different gamma ─────────────────────────────

gammas = [0.5, 0.9, 0.99]
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, g in zip(axes, gammas):
    V_g, _ = value_iteration(P, R, gamma=g)
    horizon = 1 / (1 - g) if g < 1 else float('inf')
    plot_value_heatmap(
        V_g, GRID_SIZE,
        title=f"\u03B3 = {g} (horizon \u2248 {horizon:.0f} steps)",
        cmap="viridis",
        goal_pos=state_to_pos(GOAL_STATE),
        ax=ax
    )

fig.suptitle("Effect of Discount Factor on Value Landscape", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

With $\gamma = 0.5$, value is concentrated near the goal — the agent is myopic. With $\gamma = 0.99$, value spreads across the entire grid — even distant states "know" about the goal. This is the Bellman equation's recursive magic: information about the goal propagates backward through state space.

**Question:** What happens when $\gamma = 0$? The agent only cares about immediate reward. $V(s) = 0$ for all non-goal states. The agent has no reason to move toward the goal unless it happens to be adjacent.

In [ ]:
# ── Apply Value Iteration to neuro-nav Four Rooms ────────────────────────────
# Now let's scale up to a real environment.

env = GridEnv(template=GridTemplate.four_rooms, size=GridSize.small, obs_type=GridObservation.index)
obs = env.reset()
grid_size = env.grid_size
n_states = grid_size ** 2
n_actions = 4

# Save goal position immediately after reset, BEFORE any env.step() calls
goal_pos = list(env.objects['rewards'].keys())[0]

# Build the transition matrix by probing the environment
# For each (state, action), we step the environment and record where we end up
P_env = np.zeros((n_states, n_actions, n_states))
R_env = np.zeros(n_states)

goal_state = goal_pos[0] * grid_size + goal_pos[1]
R_env[goal_state] = 1.0

for s in range(n_states):
    r, c = s // grid_size, s % grid_size
    for a in range(n_actions):
        # Manually set agent position and step
        env.reset()
        env.agent_pos = np.array([r, c])
        try:
            obs, reward, done, info = env.step(a)
            next_pos = tuple(env.agent_pos)
            next_s = next_pos[0] * grid_size + next_pos[1]
            P_env[s, a, next_s] = 1.0
        except:
            P_env[s, a, s] = 1.0  # stay in place if step fails

# Run value iteration
V_env, _ = value_iteration(P_env, R_env, gamma=0.99)

plot_value_heatmap(
    V_env, grid_size,
    title="Optimal V*(s) for Four Rooms\n(computed via Value Iteration)",
    cmap="inferno",
    goal_pos=goal_pos,
    show_values=False
)
plt.tight_layout()
plt.show()

In [ ]:
# ── Dual Perspective: Value Function vs. Expected Free Energy ─────────────────

dual_perspective_box(
    rl_text=(
        "The value function V(s) = E[\u03A3 \u03B3^t r_t] answers: 'How much cumulative "
        "reward can I expect from state s?' The Bellman equation decomposes this "
        "recursively: V(s) = R(s) + \u03B3 \u03A3 P(s'|s,a) V(s'). Value iteration "
        "computes V* by repeated application of this recursion until convergence."
    ),
    aif_text=(
        "Expected Free Energy G(\u03C0) answers: 'How much surprise and uncertainty "
        "will policy \u03C0 produce?' It decomposes into pragmatic value (do I reach "
        "preferred observations?) and epistemic value (do I reduce uncertainty?). "
        "Both are recursive evaluations of future states — but AIF minimizes "
        "expected surprise rather than maximizing expected reward."
    ),
    title="Value Function V(s) vs. Expected Free Energy G(\u03C0)"
)

## 2.6 Why Dynamic Programming Works (and When It Doesn't)

Value iteration works because of two properties:

1. **The Bellman operator is a contraction:** Each iteration brings $V$ closer to $V^*$ by at least a factor of $\gamma$. This guarantees convergence.

2. **The Markov property:** The value of a state depends only on the current state, not on history. This makes the recursion valid.

But dynamic programming has severe limitations:

- **Requires a known model:** We need $P(s'|s,a)$ and $R(s,a,s')$ explicitly. This is why we had to probe the environment above.
- **Curse of dimensionality:** The state space grows exponentially with the number of state variables. A 100x100 grid has 10,000 states; a robot with 10 joints each discretized to 100 positions has $100^{10} = 10^{20}$ states.

This is exactly why **TD learning** (Module 3) exists: it can learn values from experience *without* knowing the transition model. And it is why **Active Inference** is interesting: it builds a compact generative model and uses message passing for inference, potentially scaling better than tabular methods.

## 2.7 Exercises

### Exercise 1: Policy Iteration
Implement **policy iteration**, which alternates between:
- Policy evaluation: compute $V^\pi$ for the current policy
- Policy improvement: update $\pi$ to be greedy w.r.t. $V^\pi$

Compare the number of iterations to convergence with value iteration.

In [ ]:
# ── Exercise 1: Policy Iteration ─────────────────────────────────────────────

def policy_evaluation(policy, P, R, gamma=0.99, theta=1e-8, max_iters=1000):
    """Evaluate a fixed policy: compute V^pi."""
    n_states = P.shape[0]
    V = np.zeros(n_states)
    for i in range(max_iters):
        V_new = np.zeros(n_states)
        for s in range(n_states):
            a = policy[s]
            V_new[s] = np.sum(P[s, a, :] * (R + gamma * V))
        if np.max(np.abs(V_new - V)) < theta:
            break
        V = V_new
    return V

def policy_iteration(P, R, gamma=0.99):
    """Policy iteration: alternate evaluation and improvement."""
    n_states, n_actions = P.shape[0], P.shape[1]
    policy = np.zeros(n_states, dtype=int)  # start with action 0 everywhere
    
    for iteration in range(100):
        # Evaluate current policy
        V = policy_evaluation(policy, P, R, gamma)
        # Improve: make policy greedy w.r.t. V
        new_policy = extract_policy(V, P, R, gamma)
        
        if np.array_equal(new_policy, policy):
            print(f"Policy iteration converged after {iteration+1} improvement steps")
            return V, policy
        policy = new_policy
    
    return V, policy

V_pi, pi_star = policy_iteration(P, R, gamma=GAMMA)
print(f"V(s=0) via policy iteration: {V_pi[0]:.4f}")
print(f"V(s=0) via value iteration:  {V_star[0]:.4f}")
print(f"Difference: {abs(V_pi[0] - V_star[0]):.2e}")

### Exercise 2: Adding Walls

Modify the 4x4 grid to add walls (states the agent cannot enter). How does the value function change? Observe how value must "flow around" obstacles.

In [ ]:
# ── Exercise 2: Grid with walls ──────────────────────────────────────────────
# Add walls at positions (1,1) and (1,2) — the agent can't enter these cells

WALLS = {(1, 1), (1, 2)}

P_walls = np.zeros((N_STATES, N_ACTIONS, N_STATES))
for s in range(N_STATES):
    r, c = state_to_pos(s)
    if (r, c) in WALLS:
        P_walls[s, :, s] = 1.0  # can't leave a wall (shouldn't be here anyway)
        continue
    for a in range(N_ACTIONS):
        dr, dc = action_deltas[a]
        nr, nc = r + dr, c + dc
        if 0 <= nr < GRID_SIZE and 0 <= nc < GRID_SIZE and (nr, nc) not in WALLS:
            P_walls[s, a, pos_to_state(nr, nc)] = 1.0
        else:
            P_walls[s, a, s] = 1.0  # bounce off wall

V_walls, _ = value_iteration(P_walls, R, gamma=GAMMA)

# Set wall values to NaN for display
V_display = V_walls.copy()
for (r, c) in WALLS:
    V_display[pos_to_state(r, c)] = np.nan

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
plot_value_heatmap(V_star, GRID_SIZE, title="V* (no walls)", cmap="viridis",
                   goal_pos=state_to_pos(GOAL_STATE), ax=ax1)
plot_value_heatmap(V_display, GRID_SIZE, title="V* (with walls at (1,1) and (1,2))", 
                   cmap="viridis", goal_pos=state_to_pos(GOAL_STATE), ax=ax2)
# Mark walls
for (r, c) in WALLS:
    ax2.add_patch(plt.Rectangle((c-0.5, r-0.5), 1, 1, fill=True, 
                                 color='gray', alpha=0.8))
    ax2.text(c, r, 'WALL', ha='center', va='center', fontsize=8, color='white')

plt.tight_layout()
plt.show()

### Exercise 3: The Limit Cases of Gamma

Experiment with extreme discount factors:
- What happens when $\gamma = 0$? (myopic agent)
- What happens when $\gamma \to 1$? (far-sighted agent)
- At what $\gamma$ does the agent first learn to navigate to the goal from the farthest corner?

Try values: $\gamma \in \{0.0, 0.3, 0.5, 0.7, 0.9, 0.95, 0.99\}$.

In [ ]:
# ── Exercise 3: Explore gamma values ─────────────────────────────────────────

gammas_explore = [0.0, 0.3, 0.7, 0.95]
fig, axes = plt.subplots(1, len(gammas_explore), figsize=(5*len(gammas_explore), 4.5))

for ax, g in zip(axes, gammas_explore):
    V_g, _ = value_iteration(P, R, gamma=g)
    # Can the agent reach the goal from state 0?
    can_reach = V_g[0] > 0.001
    plot_value_heatmap(
        V_g, GRID_SIZE,
        title=f"\u03B3={g}\nV(start)={V_g[0]:.4f}\nReaches goal: {'Yes' if can_reach else 'No'}",
        cmap="viridis",
        goal_pos=state_to_pos(GOAL_STATE),
        ax=ax
    )

plt.tight_layout()
plt.show()

## 2.8 Looking Ahead

We now have the mathematical foundation:

- The **Bellman equation** recursively defines the value of a state
- **Value iteration** computes the optimal value function by iterating the Bellman operator
- **Policy extraction** derives the optimal actions from the value function
- **The discount factor** controls the planning horizon

But dynamic programming requires a *known* model: $P(s'|s,a)$. In Module 3, we will see how **TD learning** removes this requirement by learning from experience. We will also connect TD errors to **dopamine** — one of the most celebrated results in computational neuroscience.

**Next:** [Module 3: Dopamine, Prediction Errors, and TD Learning](03_td_learning.ipynb)

---

## Further Reading

1. **Bellman, R.E. (1957).** *Dynamic Programming.* Princeton University Press. The original monograph. Chapter 3 derives the equation that bears Bellman's name.

2. **Sutton, R.S. & Barto, A.G. (2018).** *Reinforcement Learning: An Introduction* (2nd ed.), Chapters 3-4. The definitive textbook treatment of MDPs, value functions, and dynamic programming.

3. **Puterman, M.L. (1994).** *Markov Decision Processes: Discrete Stochastic Dynamic Programming.* Wiley. The rigorous mathematical treatment — proves contraction and convergence.

4. **Silver, D. (2015).** *RL Course Lecture 2-3.* UCL. Available on YouTube. Excellent visual explanations of value iteration and policy iteration.

5. **Da Costa, L. et al. (2020).** *Active Inference on Discrete State-Spaces: A Synthesis.* Journal of Mathematical Psychology. Section 2 shows how AIF's expected free energy parallels the Bellman recursion.